In [1]:
import os
import pandas as pd
import numpy as np

# cmfe(기존 LTAU 기반)와 cue4unc(MC Dropout / Deep Ensemble)의 결과를 함께 비교한다.
# 세 결과 모두 동일한 CUE 아키텍처 + 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며,
# 유일한 차이는 modality 불확실성을 계산하는 방법(u_m(x))이다:
#   - cmfe:            LTAU (loss-trajectory 기반 k-NN 불확실성)
#   - cue4unc/MC_Dropout:  MC Dropout (T회 확률적 forward pass의 예측 분산)
#   - cue4unc/Deep_Ensemble: Deep Ensemble (서로 다른 seed로 학습한 독립 모델들의 예측 분산)
# scaffold_results: DeepChem ScaffoldSplitter로 나눈 5-fold 결과를 비교한다
# (run_cue_scaffold_all_*.sh / run_cue_scaffold_mc_dropout.sh / run_cue_scaffold_ensemble.sh 참고).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/scaffold_results'
cue4unc_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4unc/scaffold_results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 세 가지 소스: (표시 이름, metrics.csv 루트 경로)
# 실제 결과 폴더명은 여전히 'CUE'이므로 경로는 그대로 두고, 표시 이름만 CUE로 바꾼다.
model_sources = {
    'LTAU': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
    'MC Dropout': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v1', 'CUE'),
    'Deep Ensemble': os.path.join(cue4unc_base, 'Deep_Ensemble', '{dataset}', '{target}', 'v1', 'CUE'),
}

# 그래프에 표시될 모델 순서
model_order = ['MC Dropout', 'Deep Ensemble', 'LTAU']

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue4unc의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    cmfe와 cue4unc 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")

✅ 데이터 로드 완료: 24개의 항목이 수집되었습니다.


In [3]:
def build_rank_matrix(df_all, metric, model_order=model_order):
    """Dataset-Target 조합(행) x 모델(열)로 구성된 순위 행렬을 만든다.

    Args:
        df_all (pd.DataFrame): load_metric()으로 수집한 전체 결과(df_total).
        metric (str): 순위를 매길 metric 하나 ('rmse' | 'r2' | 'pcc').
        model_order (list[str]): 표의 컬럼 순서.

    Returns:
        pd.DataFrame: 컬럼이 'Dataset-Target' | model_order... 인 표.
            각 Dataset-Target 조합에 대해 model_order 순서로 순위를 매기고(1이 최고),
            마지막 행 'avg'는 모든 조합에 대한 순위의 평균이다.
    """
    rows = {}
    for d_name in datasets:
        for target in targets:
            df_sub = df_all[(df_all['Dataset'] == d_name) & (df_all['Target'] == target)]
            df_sub = df_sub.set_index('Model').reindex(model_order)
            if metric not in df_sub.columns or df_sub[metric].isna().all():
                continue
            ranks = df_sub[metric].rank(ascending=LOWER_IS_BETTER.get(metric, True), method='min')
            rows[f'{d_name}-{target}'] = ranks

    rank_df = pd.DataFrame(rows).T
    rank_df = rank_df.reindex(columns=model_order)
    rank_df.loc['avg'] = rank_df.mean(axis=0).round(1)
    return rank_df.reset_index().rename(columns={'index': 'Dataset-Target'})


table_save_dir = os.path.join(cue_base, 'table', 'scaffold_uncertainty_comparison_v2')
os.makedirs(table_save_dir, exist_ok=True)

for metric in metrics_of_interest:
    rank_table = build_rank_matrix(df_total, metric)
    print(f"\n=== Rank table ({metric}) ===")
    print(rank_table.to_string(index=False))
    save_path = os.path.join(table_save_dir, f'{metric}.csv')
    rank_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"✅ 저장 완료: {save_path}")


=== Rank table (rmse) ===
Dataset-Target  MC Dropout  Deep Ensemble  LTAU
         Kd-PI         3.0            2.0   1.0
       Kd-Ssel         3.0            2.0   1.0
Kd-WS(alpha=2)         2.0            3.0   1.0
    Kd-RS(k=2)         3.0            2.0   1.0
         Ki-PI         1.0            3.0   2.0
       Ki-Ssel         1.0            3.0   2.0
Ki-WS(alpha=2)         1.0            3.0   2.0
    Ki-RS(k=2)         2.0            3.0   1.0
           avg         2.0            2.6   1.4
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/table/scaffold_uncertainty_comparison_v2/rmse.csv

=== Rank table (r2) ===
Dataset-Target  MC Dropout  Deep Ensemble  LTAU
         Kd-PI         3.0            2.0   1.0
       Kd-Ssel         3.0            1.0   2.0
Kd-WS(alpha=2)         2.0            3.0   1.0
    Kd-RS(k=2)         3.0            2.0   1.0
         Ki-PI         1.0            3.0   2.0
       Ki-Ssel         1.0            2.0   3.0
Ki-WS(alpha=2)   